# 권현성의 ICT Plan & Execute Agent

- **핵심 개념:** 질문 계획, 멀티 도구 검색, 멀티턴 문맥, 근거 통합
- **나의 수정:** 검색 단위·히스토리 범위·응답 구조와 비교 질문을 개인 연구 흐름에 맞게 조정
- API 키는 코드에 저장하지 않고 Colab `Secrets`의 `OPENAI_API_KEY`를 사용합니다.


In [ ]:
!pip install langchain langchain_openai langchain_community pypdf faiss-cpu

In [ ]:
# 필요한 라이브러리 임포트
import os
import json
from typing import List, Dict, Any, Optional, Tuple
from IPython.display import display, Markdown

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate

In [ ]:
import os
from google.colab import userdata

api_key = userdata.get("OPENAI_API_KEY")
if not api_key:
    raise ValueError("Colab 왼쪽의 Secrets에 OPENAI_API_KEY를 등록하세요.")

os.environ["OPENAI_API_KEY"] = api_key


## 1. PDF 파일 다운로드

In [ ]:
# URL에서 PDF 파일 다운로드
import urllib.request

# PDF 다운로드 함수
def download_pdf_from_url(url: str, output_filename: str):
    print(f"PDF 다운로드 시작: {url}")
    urllib.request.urlretrieve(url, filename=output_filename)
    print(f"PDF 다운로드 완료: {output_filename}")

# 일본 ICT PDF 다운로드
japan_pdf_url = "https://raw.githubusercontent.com/llama-index-tutorial/llama-index-tutorial/main/ch06/ict_japan_2024.pdf"
japan_pdf_path = "japan_ict_2024.pdf"
download_pdf_from_url(japan_pdf_url, japan_pdf_path)

# 미국 ICT PDF 다운로드
usa_pdf_url = "https://raw.githubusercontent.com/llama-index-tutorial/llama-index-tutorial/main/ch06/ict_usa_2024.pdf"
usa_pdf_path = "usa_ict_2024.pdf"
download_pdf_from_url(usa_pdf_url, usa_pdf_path)

In [ ]:
# PDF 파일로부터 벡터 DB 생성 함수
def create_vectorstore_from_pdf(pdf_path: str, db_name: str) -> FAISS:
    print(f"PDF 로딩 시작: {pdf_path}")

    # PDF 로드 및 분할
    loader = PyPDFLoader(pdf_path)
    doc_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=80)
    docs = loader.load_and_split(doc_splitter)

    print(f"PDF 로딩 완료: {len(docs)}개 청크 생성됨")

    # 임베딩 및 벡터스토어 생성
    embedding = OpenAIEmbeddings(model="text-embedding-3-large")
    vectorstore = FAISS.from_documents(docs, embedding)

    # 벡터스토어 저장
    persist_directory = f"./DB/{db_name}"
    os.makedirs(persist_directory, exist_ok=True)
    vectorstore.save_local(persist_directory)

    print(f"{db_name} 벡터스토어 생성 완료")
    return vectorstore

이 함수는 PDF 파일을 로드하여 검색 가능한 벡터 데이터베이스를 생성합니다.

주요 기능:
1. PDF 파일 로딩: PyPDFLoader를 사용하여 PDF 파일의 텍스트를 추출합니다.

2. 텍스트 분할: RecursiveCharacterTextSplitter를 사용하여 추출된 텍스트를 300자 크기의
  청크로 분할하며, 인접 청크 간에 100자의 중복을 허용합니다. 이 중복은 문맥 연속성을
  유지하는 데 중요합니다.

3. 벡터 임베딩 생성: OpenAI의 text-embedding-3-large 모델을 사용하여 각 텍스트 청크를
  고차원 벡터 공간에 표현합니다. 이 임베딩은 의미적 유사도 검색의 기반이 됩니다.

4. FAISS 벡터스토어 생성: Facebook AI의 FAISS 라이브러리를 사용하여 임베딩된 벡터들을
  효율적으로 저장하고 검색할 수 있는 인덱스를 구축합니다.

5. 벡터스토어 저장: 생성된 벡터스토어를 로컬 디렉토리에 저장하여 나중에 다시 로드할 수
  있게 합니다. 이는 매번 임베딩을 다시 계산하지 않아도 되므로 시간을 절약할 수 있습니다.

각 단계마다 진행 상황을 출력하여 처리 과정을 모니터링할 수 있습니다.
함수는 최종적으로 생성된 FAISS 벡터스토어 객체를 반환하며, 이는 이후 유사도 기반
검색에 사용됩니다.

In [ ]:
# 이제 다운로드된 PDF 파일을 사용하여 벡터스토어 생성
japan_ict_db = create_vectorstore_from_pdf(japan_pdf_path, "japan_ict")
usa_ict_db = create_vectorstore_from_pdf(usa_pdf_path, "usa_ict")

## 2. 도구 생성

In [ ]:
# 도구 클래스 정의
class Tool:
    def __init__(self, name: str, description: str, vectorstore=None):
        self.name = name
        self.description = description
        self.vectorstore = vectorstore

    def __str__(self):
        return f"{self.name}: {self.description}"

In [ ]:
# 테스트로 도구 인스턴스 생성해보기
test_tool = Tool("test", "테스트 도구입니다")
print(test_tool)

In [ ]:
# 도구 설정
tools = {
    "japan_ict": Tool(
        name="japan_ict",
        description="일본의 ICT 시장동향 정보를 제공합니다. 일본 ICT와 관련된 질문은 해당 도구를 사용하세요.",
        vectorstore=japan_ict_db
    ),
    "usa_ict": Tool(
        name="usa_ict",
        description="미국의 ICT 시장동향 정보를 제공합니다. 미국 ICT와 관련된 질문은 해당 도구를 사용하세요.",
        vectorstore=usa_ict_db
    ),
    "no_tool": Tool(
        name="no_tool",
        description="사용할 도구가 없을 경우에는 기본 LLM 답변을 작성할 것입니다. 사용자의 질문을 그대로 전달하세요."
    )
}

In [ ]:
# 도구 목록 확인
for name, tool in tools.items():
    print(f"{name}: {tool.description}")

이 코드는 RAG 시스템에서 사용할 도구들을 정의하고 생성합니다:

1. Tool 클래스:
   - 도구 객체를 만들기 위한 틀(클래스)을 정의합니다.
   - name, description, vectorstore 속성을 가집니다.
   - 이 클래스로 만든 도구들은 나중에 LLM이 질문에 맞는 도구를 선택할 때 사용됩니다.

2. tools 딕셔너리:
   - Tool 클래스로 만든 도구 객체들을 저장하는 딕셔너리입니다.
   - 'japan_ict', 'usa_ict', 'no_tool' 세 가지 도구를 만들어 저장합니다.
   - 이 도구들의 description은 LLM에게 각 도구의 용도를 알려주는 역할을 합니다.
   - LLM은 사용자 질문을 분석하여 이 중에서 가장 적합한 도구를 선택하게 됩니다.
   - 예: '일본 ICT 현황'에 관한 질문이 오면 LLM은 'japan_ict' 도구를 선택합니다.

이 코드는 이후에 ToolPlanGenerator에 전달되어, LLM이 질문에 적합한 도구를 선택하고
해당 도구의 vectorstore를 사용해 관련 정보를 검색하는 기반이 됩니다.

## 3. 계획 테스트

In [ ]:
# =============================================================
# 3. 계획 생성기 (최근 2턴 히스토리 기반 멀티턴)
# =============================================================
class ToolPlanGenerator:
    def __init__(self, tools: Dict[str, Tool], model_name="gpt-4.1", temperature=0):
        self.tools = tools
        self.llm = ChatOpenAI(temperature=temperature, model_name=model_name, max_tokens=1500)

        self.template = """
        당신은 권현성의 ICT 리서치 질문을 분석하여 적절한 도구와 검색 쿼리를 결정하는 계획자입니다.

        # 사용 가능한 도구
        {tool_descriptions}

        # 대화 히스토리
        {conversation_history}

        # 현재 질문
        사용자 질문: {current_query}

        # 분석 지침
        1. 현재 질문을 분석하기 전에 반드시 대화 히스토리를 먼저 정독하세요.
        2. 현재 질문에서 불완전하거나 모호한 표현이 있으면, 반드시 대화 히스토리의 질문과 답변 내용을 근거로 구체적인 의미를 확정한 뒤 쿼리를 생성하세요.
        3. 특히 히스토리의 어시스턴트 답변 본문에 언급된 구체적인 국가명, 기술명, 사업명 등을 현재 질문과 대조하여 정확한 대상을 특정하세요.
        4. 질문에 여러 개의 요청이 포함되어 있다면, 각각에 대해 별도의 쿼리를 생성하세요.
        5. 각 쿼리가 어떤 도구를 사용해야 하는지 결정하세요.
        6. 질문이 어떤 도구와도 관련이 없다면 'no_tool'을 선택하세요.
        7. 마크다운 형식으로 작성하지 마세요.

        # 출력 형식
        반드시 다음 JSON 형식으로 출력하세요:
        {{
          "plan": {{
            "도구이름1": ["쿼리1", "쿼리2", ...],
            "도구이름2": ["쿼리3", "쿼리4", ...],
            ...
          }}
        }}
        """

        self.prompt = PromptTemplate(
            input_variables=["tool_descriptions", "conversation_history", "current_query"],
            template=self.template
        )

    def generate_plan(self, conversation_history: str, current_query: str) -> Dict:
        tool_descriptions = "\n".join([f"- {name}: {tool.description}" for name, tool in self.tools.items()])

        try:
            formatted_prompt = self.prompt.format(
                tool_descriptions=tool_descriptions,
                conversation_history=conversation_history if conversation_history else "없음",
                current_query=current_query
            )

            llm_response = self.llm.invoke(formatted_prompt)
            llm_content = llm_response.content
            print(f"LLM 응답: {llm_content}")

            try:
                plan_json = json.loads(llm_content)
                if "plan" not in plan_json:
                    return {"plan": {"no_tool": [current_query]}}
                return plan_json
            except json.JSONDecodeError as json_err:
                print(f"JSON 파싱 오류: {str(json_err)}")
                return {"plan": {"no_tool": [current_query]}}
        except Exception as e:
            print(f"계획 생성 중 오류 발생: {str(e)}")
            return {"plan": {"no_tool": [current_query]}}

# 계획 생성기 테스트
planner = ToolPlanGenerator(tools)
test_plan = planner.generate_plan("", "일본 ICT 시장 동향이 궁금해")
print("생성된 계획:")
print(json.dumps(test_plan, indent=2, ensure_ascii=False))

ToolPlanGenerator 클래스는 사용자 질문을 분석하여 어떤 도구를 사용할지, 어떤 쿼리로 검색할지 계획을 생성합니다.

주요 기능:
- 초기화 단계에서 도구 목록을 받아 각 도구의 설명을 프롬프트에 포함시킵니다.
- ChatOpenAI 모델(기본값: gpt-4.1)을 사용하여 질문 분석을 수행합니다.
- 프롬프트 템플릿은 LLM에게 도구 선택과 쿼리 생성 방법을 안내합니다.
- 멀티턴 대화를 지원하기 위해 최근 2턴의 대화 히스토리를 참고합니다.
- 멀티쿼리 생성을 위해 여러 요청이 포함된 질문을 분리하도록 지시합니다.

generate_plan 메서드:
- 대화 히스토리(conversation_history)와 현재 질문(current_query)을 입력으로 받습니다.
- 도구 설명을 포함한 프롬프트를 구성하여 LLM에 전달합니다.
- LLM은 JSON 형식으로 계획을 반환합니다: {"plan": {"도구이름": ["쿼리1", "쿼리2"]}}
- 오류 처리 로직이 포함되어 있어 LLM이 잘못된 응답을 반환하거나 JSON 파싱에 실패하면 기본값으로 no_tool을 사용합니다.

예를 들어:
- "일본 ICT 상황이 궁금해" → {"plan": {"japan_ict": ["일본 ICT 시장 현황"]}}
- "미국 ICT 신사업과 일본의 정책은?" → {"plan": {"usa_ict": ["미국 ICT 신사업"], "japan_ict": ["일본 ICT 정책"]}}
- "너는 누구니?" → {"plan": {"no_tool": ["너는 누구니?"]}}

이 클래스는 RAG 시스템의 핵심 부분으로, 사용자 질문에 따라 적절한 정보 소스를 선택하는 의사결정을 담당합니다.

실제로 잘 동작하는지 테스트 해봅시다.  

이전 질문과 이전 답변은 없다고 가정하고, "일본 ICT 시장 동향이 궁금해"라는 질문을 넣었을 때 이 도구가 계획을 어떻게 짜는지 결과를 봅시다.

In [ ]:
# 계획 생성기 테스트
planner = ToolPlanGenerator(tools)
test_plan = planner.generate_plan("", "일본 ICT 시장 동향이 궁금해")
print("생성된 계획:")
print(json.dumps(test_plan, indent=2, ensure_ascii=False))

## 4. 도구 선택 능력 + 멀티턴 + 멀티 쿼리 시스템

In [ ]:
class MultiToolRAG:
    def __init__(self, tools: Dict[str, Tool], model_name="gpt-4.1", temperature=0.2, max_history_turns=3):
        self.tools = tools
        self.tool_planner = ToolPlanGenerator(tools, model_name)
        self.llm = ChatOpenAI(temperature=temperature, model_name=model_name)
        self.max_history_turns = max_history_turns

        self.conversation_history: List[Dict[str, str]] = []

        self.response_template = """
        당신은 권현성의 ICT 리서치 어시스턴트입니다.

        # 대화 히스토리
        {conversation_history}

        # 현재 질문
        사용자 질문: {query}

        # 검색된 정보
        {context}

        # 지침
        1. 대화 히스토리를 참고하여 현재 질문의 맥락을 정확히 이해하세요.
        2. 제공된 검색 정보를 바탕으로 사용자 질문에 답변하세요.
        3. 사용자 질문에 여러 질의가 있다면 각각에 대해 답변하세요.
        4. 제공된 정보가 충분하지 않으면 솔직히 모른다고 답변하세요.
        5. 답변은 한국어로 작성하고, 국가별 핵심 내용을 항목으로 구분하세요.
        """

        self.response_prompt = PromptTemplate(
            input_variables=["conversation_history", "query", "context"],
            template=self.response_template
        )

    def _format_history(self) -> str:
        if not self.conversation_history:
            return "없음"
        recent = self.conversation_history[-self.max_history_turns:]
        formatted = []
        for i, turn in enumerate(recent, 1):
            formatted.append(f"[턴 {i}] 사용자: {turn['query']}")
            formatted.append(f"[턴 {i}] 어시스턴트: {turn['response']}")
        return "\n".join(formatted)

    def search_with_tool(self, tool_name: str, queries: List[str], num_docs=3) -> List[Document]:
        tool = self.tools.get(tool_name)
        if not tool or tool_name == "no_tool" or not tool.vectorstore:
            return []

        all_docs = []
        seen_contents = set()

        for query in queries:
            try:
                docs = tool.vectorstore.similarity_search(query, k=num_docs)
                print(f"{tool_name} 도구로 '{query}' 검색 완료: {len(docs)}개 문서 찾음")

                for doc in docs:
                    if doc.page_content not in seen_contents:
                        seen_contents.add(doc.page_content)
                        if not hasattr(doc, 'metadata') or doc.metadata is None:
                            doc.metadata = {}
                        doc.metadata['tool'] = tool_name
                        doc.metadata['query'] = query
                        all_docs.append(doc)
            except Exception as e:
                print(f"{tool_name} 도구로 '{query}' 검색 중 오류 발생: {str(e)}")

        return all_docs

    def query(self, current_query: str) -> Dict:
        history_str = self._format_history()

        plan = self.tool_planner.generate_plan(history_str, current_query)
        print(f"생성된 계획: {json.dumps(plan, indent=2, ensure_ascii=False)}")

        all_docs = []
        for tool_name, queries in plan.get("plan", {}).items():
            tool_docs = self.search_with_tool(tool_name, queries)
            all_docs.extend(tool_docs)
            print(f"{tool_name} 도구에서 {len(tool_docs)}개 문서 검색됨")

        if not all_docs and "no_tool" not in plan.get("plan", {}):
            plan["plan"]["no_tool"] = [current_query]

        if all_docs:
            context = "\n\n".join([f"[{doc.metadata.get('tool', 'unknown')}] 문서 {i+1}:\n{doc.page_content}"
                                 for i, doc in enumerate(all_docs)])
        else:
            context = "관련 문서가 검색되지 않았습니다."

        formatted_prompt = self.response_prompt.format(
            conversation_history=history_str,
            query=current_query,
            context=context
        )

        result = self.llm.invoke(formatted_prompt)
        response = result.content

        self.conversation_history.append({
            "query": current_query,
            "response": response
        })

        return {
            "query": current_query,
            "result": response,
            "plan": plan,
            "source_documents": all_docs
        }

MultiToolRAG 클래스는 멀티턴 대화, 멀티쿼리, 다중 도구 선택을 모두 지원하는 RAG 시스템의 핵심 클래스입니다.

초기화 과정:
- 사용 가능한 도구들(tools)과 언어 모델(LLM)을 설정합니다.
- 도구 계획 생성기(ToolPlanGenerator)를 초기화합니다. 이 계획 생성기는 사용자의 질문을 분석하고 각 질의에 가장 적합한 도구를 매핑합니다.
- 대화 히스토리를 저장할 리스트(`conversation_history`)를 초기화합니다. 최근 2턴까지 프롬프트에 반영됩니다.
- 답변 작성을 위한 프롬프트 템플릿을 정의합니다. 이 템플릿은 사용자 질문과 검색된 컨텍스트를 결합하여 LLM에게 어떻게 답변을 생성해야 하는지 안내합니다.

주요 메서드:

1. _format_history:
  - 이 메서드는 `self.conversation_history` 리스트에서 최근 `self.max_history_turns`(기본값 2)턴만 꺼내어 문자열로 포맷합니다.
  - 멀티턴 대화에서 "그럼", "자세히" 등의 맥락 의존 질문에 대해, ToolPlanGenerator가 저장된 정보를 참고하여 실제 의도를 파악할 수 있도록 합니다.
  - 예를 들어, 사용자가 처음에 "일본 ICT 상황이 궁금해"라고 물으면
    - `self.conversation_history`에 `{"query": "일본 ICT 상황이 궁금해", "response": (첫 번째 질문에 대한 시스템 응답)}`이 저장되고,
    그리고 이후 "그럼 정치적인 상황은?"과 같은 질문을 받을 때, 이 정보를 활용하여 "일본의 정치적인 상황"을 문의하는 것으로 해석합니다.
  - 리스트 기반이므로 이전 방식(`self.prev_query`/`self.prev_response` 덮어쓰기)과 달리 최근 2턴의 맥락을 동시에 참고할 수 있어, 2턴 전 내용을 기반으로 한 질문도 처리할 수 있습니다.
  - 다만 `max_history_turns=2`이므로 3턴 이상 이전의 맥락은 프롬프트에 포함되지 않는 한계가 있습니다.

2. search_with_tool:
  - 이 메서드는 도구 이름(`tool_name`), 쿼리 목록(`queries`), 그리고 각 쿼리당 반환할 문서 수(`num_docs`)를 입력받아, 해당 도구의 벡터스토어에서 `similarity_search`를 실행합니다.
  - 우선, `tools` 딕셔너리에서 지정된 도구 객체를 찾으며, 도구가 존재하지 않거나 `'no_tool'`이거나 벡터스토어가 없는 경우 빈 리스트를 반환합니다.
  - `'no_tool'`은 벡터 검색이 필요 없는 일반 질문 처리를 위한 특별 도구로 취급되며, 유효한 도구인 경우 각 쿼리에 대해 검색을 진행합니다.
  - 검색된 문서들은 `seen_contents` 집합을 이용하여 중복을 제거한 후, 각 문서에 도구 이름과 검색에 사용된 쿼리 정보를 메타데이터로 추가합니다.
  - 마지막으로 중복이 제거된 문서 목록(`all_docs`)을 반환하며, 검색 중 오류가 발생해도 예외 처리를 통해 시스템이 중단되지 않고 계속 진행됩니다.

3. query (메인 메서드):
  - 이 메서드는 사용자의 질문(`current_query`)을 받아 전체 RAG 프로세스를 조율하는 중앙 컨트롤러 역할을 수행합니다.
  - 초기 단계에서 `_format_history`로 최근 2턴의 히스토리를 포맷하고, `tool_planner`의 `generate_plan` 메서드를 호출해, 히스토리와 현재 질문을 함께 분석하여, 각 도구별 실행할 쿼리 목록을 담은 도구 계획(예: `{"japan_ict": ["일본 ICT 시장 현황"], "usa_ict": ["미국 ICT 신기술 개발"]}`)을 생성합니다.
  - 생성된 계획에 따라 각 도구별로 `search_with_tool`를 실행하여 문서를 검색하고, 모든 검색 결과를 하나의 리스트(`all_docs`)로 통합합니다.
  - 만약 검색 결과가 없고 도구 계획에 `'no_tool'`이 포함되어 있지 않다면, 자동으로 `'no_tool'`을 추가하여 검색 실패 시에도 시스템이 기본 응답을 제공할 수 있도록 보장합니다.
  - 이어서, 검색된 문서들을 도구 이름 접두사와 번호가 포함된 문자열 컨텍스트로 변환하고, 이 컨텍스트와 원래 질문을 포함한 프롬프트를 구성해 LLM에 전달하여 최종 응답을 생성합니다.
  - 응답 생성 후에는 `self.conversation_history`에 현재 질문과 응답을 누적 저장하고, 최종적으로 원본 질문(`query`), 생성된 응답(`result`), 도구 계획(`plan`), 그리고 검색된 문서 목록(`source_documents`)을 포함한 딕셔너리를 반환함으로써 클라이언트가 사용된 도구와 문서를 확인할 수 있도록 합니다.

전체 작동 과정의 예:
- **첫 번째 질문 ("일본 ICT 상황이 궁금해"):**
  - 도구 계획: `{"japan_ict": ["일본 ICT 시장 현황"]}`
  - 처리: `japan_ict` 도구로 관련 문서를 검색하고, 응답 생성 후 `self.conversation_history`에 저장하여 초기 대화 맥락을 형성합니다.

- **두 번째 질문 ("그럼 ICT 정치적인 상황도 궁금하고, 한국과 협업하는지도 궁금하네. 그리고 미국 ICT 신사업 추진하는 거 있대?"):**
  - 도구 계획:
```
    {
      "japan_ict": ["일본 ICT 정치적 상황", "일본 ICT 한국 협업"],
      "usa_ict": ["미국 ICT 신사업 추진"]
    }
```
  - 처리: 두 도구에서 총 3개의 쿼리로 문서를 검색하고, 결과를 병합한 후 응답을 생성하여 대화 히스토리를 업데이트합니다.

- **세 번째 질문 ("너는 누구니?"):**
  - 도구 계획: `{"no_tool": ["너는 누구니?"]}`
  - 처리: 벡터스토어 검색 없이 LLM의 일반 지식 기반 응답이 생성되며, 단순 질의응답 방식으로 처리됩니다.

이 클래스는 RAG 시스템의 모든 구성 요소를 효과적으로 통합하여, 복잡한 질문을 처리하고 적절한 정보 소스를 활용하며, 대화의 연속성을 유지하는 강력한 질의응답 시스템을 제공합니다.

In [ ]:
# RAG 시스템 초기화
rag_system = MultiToolRAG(tools)

In [ ]:
# 대화 예시 1: 일본 ICT 관련 질문
query1 = "일본 ICT 시장의 주요 성장 분야를 요약해줘"
result1 = rag_system.query(query1)
print(f"\n질문: {query1}")
print(f"답변:\n{result1['result']}")

In [ ]:
# 대화 예시 2: 멀티쿼리 및 멀티턴
query2 = "그 나라의 정책 지원과 한국 기업의 협업 가능성도 알려줘. 미국의 신규 ICT 사업과도 비교해줘."
result2 = rag_system.query(query2)
print(f"\n질문: {query2}")
print(f"답변:\n{result2['result']}")

In [ ]:
# 대화 예시 3: 도구와 관련 없는 질문
query3 = "오늘 점심 메뉴를 추천해줘"
result3 = rag_system.query(query3)
print(f"\n질문: {query3}")
print(f"답변:\n{result3['result']}")

## 5. 챗봇 UI

In [ ]:
!pip install gradio --quiet

In [ ]:
# =============================================================
# 5. 챗봇 UI (도구 호출 로그 누적 포함)
# =============================================================
import gradio as gr

tool_log_history = []

def respond(message, chat_history):
    result = rag_system.query(message)
    bot_message = result['result']

    plan = result.get('plan', {})
    plan_inner = plan.get('plan', {})
    source_docs = result.get('source_documents', [])

    log_entry = f"━━━ 턴 {len(tool_log_history) + 1} ━━━\n"
    log_entry += f"질문: {message}\n\n"
    log_entry += f"생성된 계획:\n{json.dumps(plan, indent=2, ensure_ascii=False)}\n\n"

    for tool_name, queries in plan_inner.items():
        doc_count = len([d for d in source_docs if d.metadata.get('tool') == tool_name])
        log_entry += f"🔧 도구: {tool_name}\n"
        for q in queries:
            log_entry += f"   쿼리: {q}\n"
        log_entry += f"   검색 결과: {doc_count}건\n\n"

    # 검색된 문서 내용 미리보기
    if source_docs:
        log_entry += f"--- 검색된 문서 ({len(source_docs)}건) ---\n"
        for i, doc in enumerate(source_docs):
            tool_name = doc.metadata.get('tool', 'unknown')
            query = doc.metadata.get('query', '')
            preview = doc.page_content[:100].replace('\n', ' ')
            log_entry += f"[{tool_name}] 문서 {i+1} (쿼리: {query})\n"
            log_entry += f"   {preview}...\n\n"

    tool_log_history.append(log_entry)
    full_log = "\n".join(tool_log_history)

    chat_history.append((message, bot_message))
    return "", chat_history, full_log

with gr.Blocks() as demo:
    gr.Markdown("## 권현성의 ICT Plan & Execute Agent")

    with gr.Row():
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(label="챗봇", height=500)
            msg = gr.Textbox(label="질문해주세요!", placeholder="메시지를 입력하세요...")
            clear = gr.ClearButton([msg, chatbot])

        with gr.Column(scale=2):
            tool_log = gr.Textbox(
                label="도구 호출 로그 (누적)",
                lines=25,
                max_lines=50,
                interactive=False,
            )

    msg.submit(respond, inputs=[msg, chatbot], outputs=[msg, chatbot, tool_log])

demo.launch()